In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [34]:
# Resolve the project folder dynamically so the notebook works from any machine or workspace path.
possible_roots = [
    os.getcwd(),
    os.path.abspath(os.path.join(os.getcwd(), "..")),
    os.path.abspath(os.path.join(os.getcwd(), "..", "..")),
]

project_root = next(
    (
        root for root in possible_roots
        if os.path.exists(os.path.join(root, "Database", "Raw", "upi_transactions_2024.csv"))
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Could not find 'Database/Raw/upi_transactions_2024.csv' from the current notebook location."
    )

file_path = os.path.join(project_root, "Database", "Raw", "upi_transactions_2024.csv")
df = pd.read_csv(file_path)

In [35]:
df_clean = df.copy()

df_clean = df_clean.rename(columns={
    "transaction id": "transaction_id",
    "transaction type": "transaction_type",
    "amount (INR)": "amount_inr"
})

In [36]:
df_clean["timestamp"] = pd.to_datetime(df_clean["timestamp"], errors="coerce")

In [37]:
df_clean.columns.tolist()

['transaction_id',
 'timestamp',
 'transaction_type',
 'merchant_category',
 'amount_inr',
 'transaction_status',
 'sender_age_group',
 'receiver_age_group',
 'sender_state',
 'sender_bank',
 'receiver_bank',
 'device_type',
 'network_type',
 'fraud_flag',
 'hour_of_day',
 'day_of_week',
 'is_weekend']

In [38]:
df_clean["timestamp"].isna().sum()

np.int64(0)

In [39]:
# Validate transaction IDs
df_clean["transaction_id"].is_unique

True

In [40]:
df_clean["transaction_id"].str.match(
    r"^TXN\d{10}$"
).all()

np.True_

In [41]:
df_clean["amount_inr"] = pd.to_numeric(
    df_clean["amount_inr"],
    errors="coerce"
)

In [42]:
print("Missing:", df_clean["amount_inr"].isna().sum())
print("Zero/negative:", (df_clean["amount_inr"] <= 0).sum())

Missing: 0
Zero/negative: 0


In [43]:
# Standardize categorical text
text_columns = [
    "transaction_type",
    "merchant_category",
    "transaction_status",
    "sender_age_group",
    "receiver_age_group",
    "sender_state",
    "sender_bank",
    "receiver_bank",
    "device_type",
    "network_type",
    "day_of_week"
]

for col in text_columns:
    df_clean[col] = (
        df_clean[col]
        .astype(str)
        .str.strip()
    )

In [15]:
for col in text_columns:
    print(col, df_clean[col].unique())

transaction_type ['P2P' 'P2M' 'Bill Payment' 'Recharge']
merchant_category ['Entertainment' 'Grocery' 'Fuel' 'Shopping' 'Food' 'Other' 'Utilities'
 'Transport' 'Healthcare' 'Education']
transaction_status ['SUCCESS' 'FAILED']
sender_age_group ['26-35' '36-45' '46-55' '56+' '18-25']
receiver_age_group ['18-25' '26-35' '36-45' '46-55' '56+']
sender_state ['Delhi' 'Uttar Pradesh' 'Karnataka' 'Telangana' 'Maharashtra' 'Gujarat'
 'Rajasthan' 'Tamil Nadu' 'West Bengal' 'Andhra Pradesh']
sender_bank ['Axis' 'ICICI' 'Yes Bank' 'IndusInd' 'HDFC' 'Kotak' 'SBI' 'PNB']
receiver_bank ['SBI' 'Axis' 'PNB' 'Yes Bank' 'IndusInd' 'HDFC' 'Kotak' 'ICICI']
device_type ['Android' 'iOS' 'Web']
network_type ['4G' '5G' 'WiFi' '3G']
day_of_week ['Tuesday' 'Thursday' 'Sunday' 'Monday' 'Saturday' 'Wednesday' 'Friday']


In [16]:
df_clean["fraud_flag"].isin([0, 1]).all()

np.True_

In [17]:
df_clean["fraud_flag"] = df_clean["fraud_flag"].astype("int8")

In [18]:
# Validate weekend flag

calculated_weekend = (
    df_clean["timestamp"]
    .dt.dayofweek
    .ge(5)
    .astype("int8")
)

In [19]:
# compare the calculated weekend flag with the existing one
(
    calculated_weekend != df_clean["is_weekend"]
).sum()

np.int64(0)

In [20]:
# validate the timestamp range
calculated_hour = df_clean["timestamp"].dt.hour

(
    calculated_hour != df_clean["hour_of_day"]
).sum()

np.int64(0)

In [21]:
# vaid the day of the week
calculated_day = df_clean["timestamp"].dt.day_name()

(
    calculated_day != df_clean["day_of_week"]
).sum()

np.int64(0)

In [22]:
# duplicate rows check & validation

print(
    "Duplicate rows:",
    df_clean.duplicated().sum()
)

print(
    "Duplicate transaction IDs:",
    df_clean["transaction_id"].duplicated().sum()
)

Duplicate rows: 0
Duplicate transaction IDs: 0


In [23]:
# final cleaning validation
quality_report = {
    "row_count": len(df_clean),
    "column_count": len(df_clean.columns),
    "missing_values": int(df_clean.isna().sum().sum()),
    "duplicate_rows": int(df_clean.duplicated().sum()),
    "duplicate_transaction_ids": int(
        df_clean["transaction_id"].duplicated().sum()
    ),
    "invalid_amounts": int(
        (df_clean["amount_inr"] <= 0).sum()
    ),
    "invalid_fraud_flags": int(
        (~df_clean["fraud_flag"].isin([0, 1])).sum()
    ),
    "invalid_status": int(
        (~df_clean["transaction_status"]
         .isin(["SUCCESS", "FAILED"])).sum()
    ),
    "invalid_timestamps": int(
        df_clean["timestamp"].isna().sum()
    )
}

quality_report

{'row_count': 250000,
 'column_count': 17,
 'missing_values': 0,
 'duplicate_rows': 0,
 'duplicate_transaction_ids': 0,
 'invalid_amounts': 0,
 'invalid_fraud_flags': 0,
 'invalid_status': 0,
 'invalid_timestamps': 0}

In [33]:
output_path = "E:\\my new for the me last for the change the my life for the life only goal\project\Digital Payments Fraud Intelligence\Database\clean.csv"

df_clean.to_csv(

    output_path,
    index=False
)   

<>:1: SyntaxWarning: invalid escape sequence '\p'
<>:1: SyntaxWarning: invalid escape sequence '\p'
C:\Users\Dell\AppData\Local\Temp\ipykernel_40360\2916149555.py:1: SyntaxWarning: invalid escape sequence '\p'
  output_path = "E:\\my new for the me last for the change the my life for the life only goal\project\Digital Payments Fraud Intelligence\Database\clean.csv"
